## Import Libraries

In [ ]:
from pprint import pprint

import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchmetrics.classification import Accuracy, Precision, Recall
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.datasets import ImageFolder


## Datasets and DataLoaders

In [ ]:
class CrackDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.dataset = ImageFolder(root=root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_dir = "../data"
CrackDataset = ImageFolder(root=img_dir)

train_size = int(0.7 * len(CrackDataset))
test_size = int(0.15 * len(CrackDataset))
val_size = len(CrackDataset) - train_size - test_size

train_subdataset, test_subdataset, val_subdataset = random_split(CrackDataset, [train_size, test_size, val_size])

train_dataset = CrackDataset(root_dir=img_dir, transform=train_transforms)
test_dataset = CrackDataset(root_dir=img_dir, transform=val_transforms)
val_dataset = CrackDataset(root_dir=img_dir, transform=val_transforms)



## Flexible Neural Network

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, f_in, f_out):
        super(ConvBlock, self).__init__()

        self.f_in = f_in
        self.f_out = f_out
        
        self.convseq = nn.Sequential(
            nn.Conv2d(in_channels = f_in, out_channels = f_out, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.Dropout2d(0.3),
            nn.MaxPool2d(kernel_size = 2)
        )
        

    def forward(self, x):
        x = self.convseq(x)

        return x


In [ ]:
class FlexibleCrackSeg(nn.Module):
    def __init__(self, filters_list, train_dataset, val_dataset, test_dataset, batch_size, num_channels = 3):
        super(FlexibleCrackSeg, self).__init__()

        self.filters_list = filters_list
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.test_dataset = test_dataset
        self.batch_size = batch_size
        self.num_channels = num_channels

        self.train_loader = self.CreateLoader(dataset = self.train_dataset, batch_size = self.batch_size, shuffle = True)
        self.test_loader = self.CreateLoader(dataset = self.test_dataset, batch_size = 1, shuffle = False)
        self.val_loader = self.CreateLoader(dataset = self.val_dataset, batch_size = self.batch_size, shuffle = False)

        self.conv_blocks = nn.ModuleList([])
        in_chan = self.num_channels

        for i in range(len(filters_list)):
            self.conv_blocks.append(ConvBlock(in_chan, filters_list[i]))
            in_chan = filters_list[i]

        self.flatten = nn.Flatten()
        self.fc1 = nn.LazyLinear(out_features = 1)

    def CreateLoader(self, dataset, batch_size, shuffle=None):
        loader = DataLoader(dataset = dataset, batch_size = batch_size, shuffle = shuffle)

        return loader

    def forward(self, x):
        for block in self.conv_blocks:
            x = block(x)

        x = self.flatten(x)
        x = self.fc1(x)

        return x
            


## Training and evaluation functions

In [ ]:
def train(model, train_loader, val_loader, num_epochs, loss_function, optimizer, device):
    model.to(device)

    loss_fn = loss_function
    optimizer = optimizer

    for epoch in range(num_epochs):

        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = loss_fn()


    

